# 🧠 BraTS Exploratory Data Analysis (EDA)
### Multi-Modal Brain Tumour Segmentation (FLAIR, T1, T1ce, T2)

This notebook explores:
1. **Multi-modal MRI Characteristics**: Intensity profiles of FLAIR, T1, T1ce, and T2.
2. **Class Imbalance**: Frequency distribution across background (`0`), necrotic core (`1`), edema (`2`), and enhancing tumor (`3`).
3. **Multi-Plane Spatial Visualization**: Axial, Coronal, and Sagittal cross-sections with ground-truth overlays.

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import nibabel as nib

# Ensure data exists; if raw data is not yet downloaded, generate synthetic mock cases
raw_dir = Path("../data/raw")
if not raw_dir.exists() or len(list(raw_dir.glob("*"))) == 0:
    print("Generating sample cases for EDA exploration...")
    os.system("python ../scripts/generate_mock_brats.py --num_cases 4 --out ../data/raw")

cases = sorted([p for p in raw_dir.iterdir() if p.is_dir()])
print(f"Found {len(cases)} cases. Example: {cases[0].name}")

## 1. Multi-Modal Intensity Distributions
Comparing voxel distributions inside the intracranial cavity across all four MRI modalities.

In [ ]:
sample_case = cases[0]
cid = sample_case.name
modalities = ["flair", "t1", "t1ce", "t2"]

vols = {}
for m in modalities:
    candidates = list(sample_case.glob(f"*{m}.nii*"))
    if candidates:
        vols[m] = nib.load(str(candidates[0])).get_fdata().astype(np.float32)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
colors = ["#3498db", "#9b59b6", "#e67e22", "#1abc9c"]

for idx, m in enumerate(modalities):
    data = vols[m]
    brain_voxels = data[data > 0]
    axes[idx].hist(brain_voxels.flatten(), bins=60, color=colors[idx], alpha=0.75, density=True)
    axes[idx].set_title(f"{m.upper()} Brain Voxel Distribution", fontsize=12, fontweight="bold")
    axes[idx].set_xlabel("Voxel Intensity")
    axes[idx].set_ylabel("Density")
    axes[idx].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 2. Class Balance & Voxel Proportions
Medical segmentation datasets exhibit extreme class imbalance where background voxels occupy >98% of the volume.

In [ ]:
class_counts = {0: 0, 1: 0, 2: 0, 3: 0}
label_names = {0: "Background", 1: "Necrotic Core (NCR)", 2: "Edema (ED)", 3: "Enhancing (ET)"}

for case in cases[:5]:
    seg_candidates = list(case.glob("*seg.nii*"))
    if seg_candidates:
        seg = nib.load(str(seg_candidates[0])).get_fdata().astype(np.uint8)
        seg[seg == 4] = 3
        for cls in range(4):
            class_counts[cls] += int((seg == cls).sum())

total_voxels = sum(class_counts.values())
percentages = {cls: (count / total_voxels) * 100 for cls, count in class_counts.items()}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Log-scale bar chart
cls_labels = [label_names[k] for k in range(4)]
counts = [class_counts[k] for k in range(4)]
ax1.bar(cls_labels, counts, color=["#7f8c8d", "#e74c3c", "#2ecc71", "#f1c40f"])
ax1.set_yscale("log")
ax1.set_title("Voxel Counts per Class (Log Scale)", fontsize=12, fontweight="bold")
ax1.set_ylabel("Total Voxels")
ax1.tick_params(axis="x", rotation=25)

# Tumor subregion breakdown (excluding background)
tumor_labels = [label_names[k] for k in [1, 2, 3]]
tumor_counts = [class_counts[k] for k in [1, 2, 3]]
ax2.pie(tumor_counts, labels=tumor_labels, autopct="%1.1f%%", colors=["#e74c3c", "#2ecc71", "#f1c40f"], startangle=140)
ax2.set_title("Relative Tumor Subcompartment Proportions", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

## 3. Sample MRI Slice Overlays (Multi-Modality vs Ground Truth)
Inspecting axial slice with high tumor burden.

In [ ]:
seg_candidates = list(sample_case.glob("*seg.nii*"))
seg = nib.load(str(seg_candidates[0])).get_fdata().astype(np.uint8)
seg[seg == 4] = 3

d = seg.shape[2]
tumor_slices = [z for z in range(d) if np.any(seg[:, :, z] > 0)]
sample_z = tumor_slices[len(tumor_slices) // 2] if tumor_slices else d // 2

cmap = ListedColormap([(0, 0, 0, 0), "#e74c3c", "#2ecc71", "#f1c40f"])

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for i, m in enumerate(modalities):
    axes[i].imshow(vols[m][:, :, sample_z].T, cmap="gray", origin="lower")
    axes[i].set_title(m.upper(), fontsize=12)
    axes[i].axis("off")

# Ground truth overlay on FLAIR
axes[4].imshow(vols["flair"][:, :, sample_z].T, cmap="gray", origin="lower")
axes[4].imshow(seg[:, :, sample_z].T, cmap=cmap, vmin=0, vmax=3, origin="lower", alpha=0.6)
axes[4].set_title("FLAIR + Ground Truth", fontsize=12)
axes[4].axis("off")

plt.suptitle(f"Case {cid} - Axial Slice {sample_z}", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()